# Outlook

In this notebook, we code the DDPG algorithm, and then you will extend it
into TD3.

The DDPG algorithm is explained in [this
video](https://www.youtube.com/watch?v=0D6a0a1HTtc) and you can also read
[the corresponding
slides](http://pages.isir.upmc.fr/~sigaud/teach/ddpg.pdf).

## Setup

**On Colab**: run the installation cell below (generated automatically,
pinned versions). **Locally**: install the environment with `uv sync`, no
need for this cell.

In [ ]:
import copy
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import rl_mind.envs
from dataclasses import dataclass, replace
from scipy import stats
from torch import Tensor
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
from rl_mind.profiles import Profile
from rl_mind.core import Action, Actor
from rl_mind.nn import build_mlp, soft_update
from rl_mind.env import VecEnv
from rl_mind.data import ReplayBuffer, Transitions
from rl_mind.collectors import TransitionCollector
from rl_mind.evaluation import Evaluator
from rl_mind.notebook import run_directory, setup_tensorboard, silence_known_warnings


## The `rl_mind` toolkit

The `rl_mind` package ([documentation](https://master-dac.isir.upmc.fr/rl/rl-mind/)),
installed with the practicals, provides a small set of *typed* helpers used
throughout the RL notebooks. The ones used here:

- `VecEnv` runs several copies of a gymnasium environment in parallel,
  using torch tensors;
- `Transitions` is a batch of $(s, a, r, s', \textrm{terminated})$ tuples,
  stored in a `ReplayBuffer`;
- `TransitionCollector` runs an actor in a `VecEnv` and returns the
  corresponding `Transitions`;
- `Evaluator` periodically evaluates an actor and keeps a copy of the best
  one;
- `Actor` is the base class of all the policies: `actor(obs)` returns an
  `Action` (its training-time, exploratory behavior) while `actor.act(obs)`
  returns the action tensor of its deterministic (evaluation) behavior;
- `build_mlp` builds a multi-layer perceptron, and `soft_update` moves a
  target network towards its online network (see *Target network update*
  below).

In [ ]:
# pygame and tensorboard still use the deprecated `pkg_resources` API
silence_known_warnings()

# DDPG

The [DDPG](https://arxiv.org/pdf/1509.02971.pdf) algorithm is an
*off-policy*, actor-critic algorithm for **continuous actions**: the data
used for learning is sampled from a replay buffer that stores transitions
$(s_t, a_t, r_{t+1}, s_{t+1})$ collected by an exploratory version of the
current actor.

An important detail: a transition also stores whether $s_{t+1}$ is a
**terminal** state (`terminated`). When it is, the value of the future is
zero and the critic target must not bootstrap; when the episode was merely
*truncated* by a time limit, the agent could have continued and we do
bootstrap. This is why the target is multiplied by `~batch.terminated`
below.

## Definition of the agents

### Critic $Q(s, a)$

The critic is a neural network taking the state $s$ and action $a$ as
input; its output is a single scalar, the value $Q(s,a)$ of performing $a$
in $s$. It is a plain `torch.nn.Module`.

In [ ]:
class ContinuousQNetwork(nn.Module):
    """The Q-network $Q(s, a)$ for continuous actions"""

    def __init__(self, obs_dim: int, hidden: tuple[int, ...], action_dim: int):
        super().__init__()
        self.model = build_mlp([obs_dim + action_dim, *hidden, 1])

    def forward(self, obs: Tensor, action: Tensor) -> Tensor:
        """Compute $Q(s, a)$ for a batch: `[B, obs_dim] x [B, action_dim] -> [B]`"""
        return self.model(torch.cat([obs, action], dim=1)).squeeze(-1)

### Actor

The actor is also a neural network; it takes a state $s$ as input and
outputs an action $a = \pi(s)$ in $[-1, 1]^d$ (note the final `tanh`). As
it is deterministic, its training and evaluation behaviors coincide.

In [ ]:
class ContinuousDeterministicActor(Actor[Action]):
    def __init__(self, obs_dim: int, hidden: tuple[int, ...], action_dim: int):
        """Creates a new actor with continuous actions in `action_dim` dimensions.

        :param obs_dim: The dimension of the observation space
        :param hidden: A tuple of integers specifying the sizes of the hidden layers
        :param action_dim: The dimension of the action space
        """
        super().__init__()
        self.model = build_mlp(
            [obs_dim, *hidden, action_dim], output_activation=nn.Tanh()
        )

    def forward(self, obs: Tensor) -> Action:
        return Action(value=self.model(obs))

### Exploration

A deterministic actor does not explore. In the continuous action domain,
the usual solution is to add Gaussian noise to the output of the actor.
`GaussianNoise` *wraps* another actor: the noise is added at training time
only (`forward`), while the evaluation behavior (`act`) is that of the
wrapped actor.

In [the original DDPG paper](https://arxiv.org/pdf/1509.02971.pdf), the
authors rather used the more sophisticated Ornstein-Uhlenbeck noise, where
the noise is correlated between one step and the next. As a follow-up, you
could implement this strategy.

In [ ]:
class GaussianNoise(Actor[Action]):
    """Adds Gaussian noise to the actions of another actor (at training time)"""

    def __init__(self, actor: Actor[Action], sigma: float):
        super().__init__()
        self.actor = actor
        self.sigma = sigma

    def forward(self, obs: Tensor) -> Action:
        action = self.actor(obs).value
        return Action(value=action + self.sigma * torch.randn_like(action))

    def act(self, obs: Tensor) -> Tensor:
        return self.actor.act(obs)

## Losses

### Critic loss

The critic is learned by minimizing the squared temporal difference error
over a batch of transitions sampled from the replay buffer:

$$ L(\phi) = \mathbb{E}_{(s,a,r,s',d) \sim \mathcal{D}}\left[ \Big(
Q_\phi(s,a) - \big(r + \gamma\, (1-d)\, Q_{\phi_\text{targ}}(s',
\pi_\theta(s')) \big) \Big)^2 \right] $$

where $d$ tells whether the episode is terminated in $s'$, $\pi_\theta$ is
the (current) actor, and $Q_{\phi_\text{targ}}$ is the **target critic** (a
slowly updated copy of the critic, which stabilizes learning). The target
$r + \gamma (1-d) Q_{\phi_\text{targ}}(s', \pi_\theta(s'))$ is treated as a
constant: no gradient flows through it.

Note the link with **Double DQN**: there, the online network *selects* the
next action ($\arg\max_{a'} Q_\phi(s', a')$) and the target network
*evaluates* it. DDPG does the same, with the actor playing the role of the
$\arg\max$: the current actor selects $a' = \pi_\theta(s')$ (it is trained
to maximize $Q_\phi$), and the target critic evaluates $Q_{\phi_\text{targ}}(s',
a')$.

In [ ]:
def compute_critic_loss(
    gamma: float, batch: Transitions[Action], q_values: Tensor, next_q_values: Tensor
) -> Tensor:
    """Compute the DDPG critic loss from a batch of transitions

    :param gamma: The discount factor
    :param batch: The batch of transitions
    :param q_values: $Q(s_t, a_t)$ from the critic (shape `[B]`)
    :param next_q_values: $Q'(s_{t+1}, \\pi(s_{t+1}))$ from the target critic
        (shape `[B]`)
    :return: The critic loss (a scalar)
    """
    # Compute the target (do not bootstrap when `batch.terminated`), then the MSE loss
    target = batch.reward + gamma * (~batch.terminated).float() * next_q_values
    return F.mse_loss(q_values, target.detach())

### Actor loss

The actor loss is straightforward: we want the actor to maximize the
Q-values of its actions, thus we minimize the mean of negated Q-values:

$$ L(\theta) = -\mathbb{E}_{s \sim \mathcal{D}}\left[ Q_\phi(s,
\pi_\theta(s)) \right] $$

The gradient flows through the critic into the actor, but only the
actor's parameters $\theta$ are updated with this loss.

In [ ]:
def compute_actor_loss(q_values: Tensor) -> Tensor:
    """Return the actor loss given $Q(s_t, \\pi(s_t))$ (shape `[B]`)"""
    # Compute the actor loss
    return -q_values.mean()

### Target network update

The target critic $Q_{\phi_\text{targ}}$ of the critic loss is not
learned: if it were the critic itself, the target of the regression would
move with every gradient step, and learning would chase its own tail. As in
DQN, it is a *copy* of the critic, but rather than copying the critic every
$K$ steps, DDPG lets the copy follow the critic slowly, after each update:

$$ \phi_\text{targ} \leftarrow \tau\, \phi + (1 - \tau)\, \phi_\text{targ} $$

This is a *soft* (or Polyak) update: with $\tau = 0.05$ (`cfg.tau`), the
target critic is an exponential moving average of the recent critics,
about the last $1/\tau = 20$ of them. $\tau = 1$ would be a hard copy, and
$\tau \to 0$ a target that never moves.

`soft_update(source, target, tau)` from `rl_mind.nn` applies this formula
to each parameter of `target`, in place and without tracking gradients
([documentation](https://master-dac.isir.upmc.fr/rl/rl-mind/api/utilities.html#rl_mind.nn.soft_update)):

```python
with torch.no_grad():
    for p, p_targ in zip(source.parameters(), target.parameters()):
        p_targ.mul_(1 - tau).add_(tau * p)
```

## Hyper-parameters

All the hyper-parameters of the algorithm are gathered in a small
dataclass — typed, documented, with visible defaults.

In [ ]:
@dataclass(frozen=True)
class DDPGConfig:
    env_name: str = "CartPoleContinuous-v1"
    seed: int = 1

    #: Total number of environment steps
    max_steps: int = Profile.pick(30_000, fast_test=600)
    #: Number of parallel training environments
    n_envs: int = 1
    #: Environment steps between two gradient updates
    steps_per_update: int = 1
    #: Steps before learning starts
    learning_starts: int = Profile.pick(1_000, fast_test=100)

    #: Replay buffer capacity
    buffer_size: int = 200_000
    batch_size: int = 64

    #: Discount factor
    gamma: float = 0.98
    #: Target network update coefficient
    tau: float = 0.05
    #: Exploration noise
    action_noise: float = 0.1

    actor_hidden: tuple[int, ...] = (64, 64)
    critic_hidden: tuple[int, ...] = (64, 64)
    lr_actor: float = 1e-3
    lr_critic: float = 1e-3

    #: Steps between two evaluations, and number of evaluation episodes
    eval_interval: int = Profile.pick(2_000, fast_test=200)
    n_eval_envs: int = 10

## Main training loop

Everything is now explicit: collect transitions, store them into the
buffer, sample a batch, update the critic, update the actor, and update
the target network. Note that:

- the actions of the *next* states are recomputed with the current actor
  (`actor(batch.next_obs)`) — the actions stored in the buffer are only
  used for the critic update;
- the target computation is wrapped in `torch.no_grad()`: gradients must
  only flow through $Q_\phi(s_t, a_t)$.

In [ ]:
def run_ddpg(cfg: DDPGConfig) -> Evaluator:
    torch.manual_seed(cfg.seed)
    env = VecEnv(cfg.env_name, cfg.n_envs, seed=cfg.seed)

    # Create the actor, the critic and its target, and the optimizers

    # `env.observation_dim` and `env.action_dim` contain the dimension of resp.
    # the observation and action spaces
    actor = ContinuousDeterministicActor(
        env.observation_dim, cfg.actor_hidden, env.action_dim
    )
    critic = ContinuousQNetwork(env.observation_dim, cfg.critic_hidden, env.action_dim)
    # the target critic is a copy of the critic (`copy.deepcopy`)
    target_critic = copy.deepcopy(critic)
    target_critic.requires_grad_(False)
    # one optimizer per network (`cfg.lr_actor`, `cfg.lr_critic`)
    actor_optimizer = torch.optim.Adam(actor.parameters(), lr=cfg.lr_actor)
    critic_optimizer = torch.optim.Adam(critic.parameters(), lr=cfg.lr_critic)

    # Data collection (with exploration noise), replay buffer and evaluation
    collector = TransitionCollector(env, GaussianNoise(actor, cfg.action_noise))
    buffer = ReplayBuffer(cfg.buffer_size)
    run_dir = run_directory(f"ddpg-{cfg.env_name}-S{cfg.seed}")
    evaluator = Evaluator(
        VecEnv(cfg.env_name, cfg.n_eval_envs, seed=cfg.seed + 100),
        every=cfg.eval_interval,
        run_dir=run_dir,
        writer=SummaryWriter(run_dir),
    )

    pbar = tqdm(total=cfg.max_steps)
    while collector.steps < cfg.max_steps:
        buffer.add(collector.collect(cfg.steps_per_update))
        pbar.update(collector.steps - pbar.n)
        if len(buffer) < cfg.learning_starts:
            continue

        batch = buffer.sample(cfg.batch_size)

        # Q-values of the actions that were played (this is where gradients flow)
        q_values = critic(batch.obs, batch.action.value)

        with torch.no_grad():
            # Q-values of the *current* actor's actions in the next states, from the target critic

            # as in Double DQN: the actor selects the action, the target
            # critic evaluates it
            next_q_values = target_critic(batch.next_obs, actor(batch.next_obs).value)

        # Critic update: compute the loss, then a gradient step
        critic_loss = compute_critic_loss(cfg.gamma, batch, q_values, next_q_values)
        critic_optimizer.zero_grad()
        critic_loss.backward()
        critic_optimizer.step()

        # Actor update: maximize Q(s, pi(s)), with a gradient step on the actor only
        actor_loss = compute_actor_loss(critic(batch.obs, actor(batch.obs).value))
        actor_optimizer.zero_grad()
        actor_loss.backward()
        actor_optimizer.step()

        # Soft update of the target critic (`soft_update`)
        soft_update(critic, target_critic, cfg.tau)

        evaluator.writer.add_scalar("loss/critic", critic_loss.item(), collector.steps)
        evaluator.writer.add_scalar("loss/actor", actor_loss.item(), collector.steps)
        if result := evaluator.run_if_needed(collector.steps, actor):
            pbar.set_description(
                f"eval={result.mean:7.1f} best={evaluator.best_reward:7.1f}"
            )

    pbar.close()
    return evaluator

## Launching tensorboard to visualize the results

In [ ]:
setup_tensorboard()

In [ ]:
ddpg_config = DDPGConfig()
ddpg = run_ddpg(ddpg_config)

In [ ]:
ddpg.visualize_best()

# TD3

Starting from the above version, you should code [the TD3
algorithm](http://proceedings.mlr.press/v80/fujimoto18a/fujimoto18a.pdf).

## Differences with DDPG

There are three main differences between DDPG and TD3.

### Clipped double-Q learning

To fight the overestimation bias, the target value for the Bellman backup
takes the min over two target critics:

$$y(r, s', d) = r + \gamma (1 - d) \min_{i=1,2} {Q'}_{\phi_{i,
\text{targ}}}(s', f(s')) \qquad (1)$$

where $d$ tells whether the episode is terminated and $f(s')$ is the value
of some policy (see below) at state $s'$.

Both Q-value estimators are learned using the following loss for $i \in
\{1,2\}$:

$$L(\phi_i, {\mathcal D}) = \mathbb{E}_{(s,a,r,s',d) \sim {\mathcal D}}
{\Bigg( Q_{\phi_i}(s,a) - y(r,s',d) \Bigg)^2}. \qquad (2)$$

### Target policy smoothing

The original implementation of DDPG used a policy and a target policy, as
for critics, though it makes little mathematical sense in the case of
policies (it is not learned from a shifting regression problem). However,
using a target policy was smoothing policy learning, which is good to
fight instabilities but slows down performance. Further studies with DDPG
showed that using this target policy was not consistently improving
learning performance, and the mechanism was dropped in many standard
implementations.

As TD3 inherits from DDPG, the question of keeping the target policy is
left open. It is kept in the paper and in the Stable Baselines 3
implementation, so it seems like a good idea to keep it.

The target policy is used to update the target value for both critics as
in (2). In addition, the action from that target policy is smoothed with
additional clipped noise, to make policy learning more robust. So, in (1)
above, the function $f$ is given by:

$$f(s') = \pi_{\theta_\text{targ}}(s') + \epsilon, \quad \text{where } \epsilon \sim
\text{clip}(\mathcal{N}(0,\sigma),-c,c)$$

where $\sigma$ is the target policy noise and $c$ is the target policy
noise clip.

### Delayed policy updates

The policy is updated less frequently than the critics, using a
`policy_delay` parameter.

## Your job

Implement `run_td3` below, starting from the DDPG loop above. There are
exactly the three mechanisms described in the previous sections to add — and
nothing else:

1. **Clipped double-Q learning** — replace the single critic and its target
   with two critics (`critic_1`, `critic_2`) and their two targets. Compute a
   single target from the **minimum** of the two *target* critics, and
   regress **both** critics towards it.
2. **Target policy smoothing** — the next actions used in that target are
   given by the *target actor*, perturbed by a Gaussian noise of standard
   deviation `cfg.target_noise`, clipped to ±`cfg.target_noise_clip` (and the
   resulting action clipped to the action range).
3. **Delayed policy updates** — update the actor (by maximizing the Q-values
   of `critic_1`) only every `cfg.policy_delay` gradient steps, and softly
   update the three target networks at the same rate.

In [ ]:
@dataclass(frozen=True)
class TD3Config(DDPGConfig):
    #: Number of critic updates between two policy updates
    policy_delay: int = 2
    #: Std of the noise added to the target policy actions
    target_noise: float = 0.2
    #: Clipping of the target policy noise
    target_noise_clip: float = 0.5

In [ ]:
def run_td3(cfg: TD3Config) -> Evaluator:
    torch.manual_seed(cfg.seed)
    env = VecEnv(cfg.env_name, cfg.n_envs, seed=cfg.seed)

    actor = ContinuousDeterministicActor(
        env.observation_dim, cfg.actor_hidden, env.action_dim
    )
    target_actor = copy.deepcopy(actor)
    target_actor.requires_grad_(False)

    critic_1 = ContinuousQNetwork(env.observation_dim, cfg.critic_hidden, env.action_dim)
    critic_2 = ContinuousQNetwork(env.observation_dim, cfg.critic_hidden, env.action_dim)
    target_critic_1 = copy.deepcopy(critic_1)
    target_critic_2 = copy.deepcopy(critic_2)
    target_critic_1.requires_grad_(False)
    target_critic_2.requires_grad_(False)

    actor_optimizer = torch.optim.Adam(actor.parameters(), lr=cfg.lr_actor)
    critic_optimizer = torch.optim.Adam(
        [*critic_1.parameters(), *critic_2.parameters()], lr=cfg.lr_critic
    )

    collector = TransitionCollector(env, GaussianNoise(actor, cfg.action_noise))
    buffer = ReplayBuffer(cfg.buffer_size)
    run_dir = run_directory(f"td3-{cfg.env_name}-S{cfg.seed}")
    evaluator = Evaluator(
        VecEnv(cfg.env_name, cfg.n_eval_envs, seed=cfg.seed + 100),
        every=cfg.eval_interval,
        run_dir=run_dir,
        writer=SummaryWriter(run_dir),
    )

    n_updates = 0
    pbar = tqdm(total=cfg.max_steps)
    while collector.steps < cfg.max_steps:
        buffer.add(collector.collect(cfg.steps_per_update))
        pbar.update(collector.steps - pbar.n)
        if len(buffer) < cfg.learning_starts:
            continue

        batch = buffer.sample(cfg.batch_size)
        n_updates += 1

        q_values_1 = critic_1(batch.obs, batch.action.value)
        q_values_2 = critic_2(batch.obs, batch.action.value)

        with torch.no_grad():
            next_actions = target_actor(batch.next_obs).value
            noise = (torch.randn_like(next_actions) * cfg.target_noise).clamp(
                -cfg.target_noise_clip, cfg.target_noise_clip
            )
            next_actions = (next_actions + noise).clamp(-1.0, 1.0)
            next_q_values = torch.min(
                target_critic_1(batch.next_obs, next_actions),
                target_critic_2(batch.next_obs, next_actions),
            )

        critic_loss = compute_critic_loss(
            cfg.gamma, batch, q_values_1, next_q_values
        ) + compute_critic_loss(cfg.gamma, batch, q_values_2, next_q_values)
        critic_optimizer.zero_grad()
        critic_loss.backward()
        critic_optimizer.step()
        evaluator.writer.add_scalar("loss/critic", critic_loss.item(), collector.steps)

        if n_updates % cfg.policy_delay == 0:
            actor_loss = compute_actor_loss(critic_1(batch.obs, actor(batch.obs).value))
            actor_optimizer.zero_grad()
            actor_loss.backward()
            actor_optimizer.step()
            evaluator.writer.add_scalar("loss/actor", actor_loss.item(), collector.steps)

            soft_update(critic_1, target_critic_1, cfg.tau)
            soft_update(critic_2, target_critic_2, cfg.tau)
            soft_update(actor, target_actor, cfg.tau)

        if result := evaluator.run_if_needed(collector.steps, actor):
            pbar.set_description(
                f"eval={result.mean:7.1f} best={evaluator.best_reward:7.1f}"
            )

    pbar.close()
    return evaluator

In [ ]:
td3_config = TD3Config()
td3 = run_td3(td3_config)

In [ ]:
td3.visualize_best()

# Experimental comparison

The `history` of each `Evaluator` contains all the evaluations (steps and
rewards), which we can use to compare the two algorithms: we plot the mean
cumulated reward (± its standard deviation) along training, and run a
Welch t-test on the final evaluations.

As an exercise, take an environment where the overestimation bias may
matter (e.g. `Pendulum-v1`), and compare the performance of DDPG and TD3.
Visualize the Q-values long before convergence to see whether DDPG indeed
overestimates them with respect to TD3.

In [ ]:
def plot_evaluations(**evaluators: Evaluator):
    for name, evaluator in evaluators.items():
        steps = [result.step for result in evaluator.history]
        means = torch.tensor([result.mean for result in evaluator.history])
        stds = torch.tensor(
            [float(result.rewards.std()) for result in evaluator.history]
        )
        (line,) = plt.plot(steps, means, label=name)
        plt.fill_between(
            steps, means - stds, means + stds, alpha=0.2, color=line.get_color()
        )
    plt.xlabel("steps")
    plt.ylabel("cumulated reward")
    plt.legend()
    plt.show()


plot_evaluations(ddpg=ddpg, td3=td3)

# The Welch t-test tells us whether the difference between the final
# evaluations is statistically significant (p-value < 0.05)
welch = stats.ttest_ind(
    ddpg.history[-1].rewards.numpy(), td3.history[-1].rewards.numpy(), equal_var=False
)
print(
    f"DDPG vs TD3 (final evaluation): t={welch.statistic:.2f}, p-value={welch.pvalue:.3f}"
)